# Week 10 · Class 5 — Independent Capstone
## Historical Production Data Quality Pipeline

**Assessment notebook — not a guided teaching lesson.** Produce a self-contained, reproducible analytic report. Include your own explanations, assumptions, code outputs, and findings. This notebook must call and demonstrate the functions you implement in `src/production_pipeline.py`.

**Important:** Use the full project brief as the source of truth. Do not delete the original dataset or change input values.

## 1. Problem statement and objectives
Write 1–2 paragraphs describing why missing 6-hour observations, duplicates, incorrect measurements, and changing sensor reliability matter. State the required outputs and your policy for zeros versus missing readings.

In [3]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "production_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find src/production_pipeline.py")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.production_pipeline import (
    ingest_and_validate,
    repair_and_align,
    summarize_daily,
    run_pipeline,
)

INPUT = PROJECT_ROOT / "data" / "historical_production_capstone.csv"
OUTPUT = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Input present:", INPUT.exists())

Project root: c:\Users\abdul\OneDrive\Documents\Charlotte2026\module 2\charlotte-week10-python
Input present: True


## 2. Data ingestion and schema validation
Show raw row count, column names, data types, sample rows and invalid timestamps. Explain why timestamp parsing uses `errors="coerce"`.

In [4]:
raw_preview = pd.read_csv(INPUT)

print("Raw CSV shape:", raw_preview.shape)
print("Columns:", raw_preview.columns.tolist())
display(raw_preview.head())

valid_records, rejected_records = ingest_and_validate(INPUT)

print("Valid records:", len(valid_records))
print("Rejected records:", len(rejected_records))
display(rejected_records)

Raw CSV shape: (47, 8)
Columns: ['well_id', 'timestamp', 'oil_bbl', 'water_bbl', 'gas_mscf', 'pressure_psi', 'status', 'ingested_at']


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at
0,WELL-21,2026-04-06 18:00,77.0,22,151,2839,ACTIVE,2026-04-06 18:04
1,WELL-21,2026-04-08 00:00,68.0,24,161,2904,ACTIVE,2026-04-08 00:04
2,WELL-21,2026-04-09 00:00,68.0,22,169,2956,ACTIVE,2026-04-09 00:04
3,WELL-22,2026-04-06 00:00,78.0,24,152,2960,ACTIVE,2026-04-06 00:04
4,WELL-22,2026-04-07 00:00,78.0,25,160,not_available,ACTIVE,2026-04-07 00:04


Valid records: 46
Rejected records: 1


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at,source_line,rejection_reason
34,WELL-22,NaT,72.0,22,150,2960,ACTIVE,2026-04-10 01:00:00,36,Invalid timestamp


## 3. Deduplication, alignment, and cleaning
Show how latest-ingestion duplicate precedence works. Explain how missing *timestamps* differ from missing *values*. Display a before-and-after example for a chosen well. Explain how you prevent false oil production totals.

In [6]:
aligned_records, repair_quality = repair_and_align(valid_records)

sample_columns = [
    "well_id",
    "timestamp",
    "oil_bbl",
    "water_bbl",
    "gas_mscf",
    "pressure_psi",
    "slot_inserted",
]

print("Sample rows by well:")
display(aligned_records.groupby("well_id", sort=True).head(3)[sample_columns])

measurement_columns = [
    "oil_bbl",
    "water_bbl",
    "gas_mscf",
    "pressure_psi",
]

print("Missing values after repair:")
print(aligned_records[measurement_columns].isna().sum())

print("Quality events:")
display(repair_quality)

Sample rows by well:


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,slot_inserted
0,WELL-21,2026-04-06 00:00:00,68.0,22.0,145.0,2800.0,False
1,WELL-21,2026-04-06 06:00:00,76.0,23.0,147.0,2813.0,False
2,WELL-21,2026-04-06 12:00:00,NaN,24.0,149.0,2826.0,False
16,WELL-22,2026-04-06 00:00:00,78.0,24.0,152.0,2960.0,False
17,WELL-22,2026-04-06 06:00:00,81.0,25.0,154.0,2973.0,False
18,WELL-22,2026-04-06 12:00:00,84.0,26.0,156.0,2986.0,False
32,WELL-23,2026-04-06 00:00:00,88.0,26.0,159.0,3120.0,False
33,WELL-23,2026-04-06 06:00:00,91.0,27.0,161.0,3133.0,False
34,WELL-23,2026-04-06 12:00:00,94.0,28.0,163.0,3146.0,False


Missing values after repair:
oil_bbl         6
water_bbl       5
gas_mscf        5
pressure_psi    0
dtype: int64
Quality events:


,metric,count
0,duplicate_rows_removed,2
1,six_hour_slots_inserted,4
2,missing_oil_bbl,6
3,missing_water_bbl,5
4,missing_gas_mscf,5
5,missing_pressure_psi,0


## 4. Daily resampling / aggregation
Report daily oil, water and gas totals, observation completeness, and incomplete days. Explain why sum(min_count=1) protects all-missing groups.

In [7]:
daily_summary = summarize_daily(aligned_records)

print("Daily summary rows:", len(daily_summary))
display(daily_summary)

Daily summary rows: 12


,well_id,date,expected_slots,oil_bbl,water_bbl,gas_mscf,oil_bbl_valid_readings,oil_bbl_complete,water_bbl_valid_readings,water_bbl_complete,gas_mscf_valid_readings,gas_mscf_complete
0,WELL-21,2026-04-06,4,221.0,91.0,592.0,3,False,4,True,4,True
1,WELL-21,2026-04-07,4,219.0,68.0,469.0,3,False,3,False,3,False
2,WELL-21,2026-04-08,4,290.0,71.0,656.0,4,True,3,False,4,True
3,WELL-21,2026-04-09,4,290.0,91.0,688.0,4,True,4,True,4,True
4,WELL-22,2026-04-06,4,330.0,99.0,620.0,4,True,4,True,4,True
5,WELL-22,2026-04-07,4,243.0,75.0,486.0,4,True,4,True,4,True
6,WELL-22,2026-04-08,4,159.0,76.0,512.0,2,False,3,False,3,False
7,WELL-22,2026-04-09,4,330.0,99.0,716.0,4,True,4,True,4,True
8,WELL-23,2026-04-06,4,273.0,81.0,483.0,3,False,3,False,3,False
9,WELL-23,2026-04-07,4,370.0,108.0,680.0,4,True,4,True,4,True


## 5. Validation evidence and interpretation
Provide results of tests you wrote, expected-vs-actual checks for row counts, and at least three operational insights or limitations. Note that the data are synthetic.

In [8]:
pipeline_results = run_pipeline(INPUT, OUTPUT)

output_files = [
    "cleaned_production.csv",
    "data_quality_report.csv",
    "daily_production_summary.csv",
    "rejected_source_records.csv",
]

for filename in output_files:
    print(filename, "exists:", (OUTPUT / filename).exists())

cleaned_production.csv exists: True
data_quality_report.csv exists: True
daily_production_summary.csv exists: True
rejected_source_records.csv exists: True


## 6. Conclusions and reproducibility
Explain your cleaning decisions, their impact on interpretation, and how another analyst can rerun the pipeline. Include a link to the GitHub Pull Request in your README (do not hard-code a private token).